<a href="https://colab.research.google.com/github/Faiq-danZ/worksafe-ai/blob/main/training/train_tabular.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np
import os

print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.19.0


In [9]:
def weighted_risk_loss(y_true, y_pred):
    # Mean Squared Error dasar
    mse = tf.square(y_true - y_pred)
    weight = tf.where(y_true > 0.5, 2.0, 1.0)
    return tf.reduce_mean(mse * weight)

print("Custom Loss Function berhasil didefinisikan.")

Custom Loss Function berhasil didefinisikan.


In [4]:
def build_tabular_model(input_shape):
    # Menggunakan Input Layer (Ciri khas Functional API)
    inputs = tf.keras.Input(shape=(input_shape,))

    # Layer pertama dengan Batch Normalization
    x = layers.Dense(64, activation='relu')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.2)(x)

    # Layer kedua
    x = layers.Dense(32, activation='relu')(x)

    # Output layer (Sigmoid menghasilkan skor antara 0 sampai 1)
    outputs = layers.Dense(1, activation='sigmoid')(x)

    # Inisialisasi Model
    model = models.Model(inputs=inputs, outputs=outputs, name="WorkSafe_Tabular_Model")

    # Compile
    model.compile(
        optimizer='adam',
        loss=weighted_risk_loss,
        metrics=['mae', 'accuracy']
    )
    return model

# Inisialisasi model dengan asumsi 10 fitur input
model = build_tabular_model(input_shape=10)
model.summary()

Model: "WorkSafe_Tabular_Model"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 10)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 64)             │           704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,073 (12.00 KB)

 Trainable params: 2,945 (11.50 KB)

 Non-trainable params: 128 (512.00 B)

In [5]:
class RiskMonitoringCallback(tf.keras.callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):

        if logs.get('mae') is not None and logs.get('mae') < 0.1:
            print(f"\n[INFO] Target MAE tercapai di epoch {epoch+1}. Menghentikan training!")
            self.model.stop_training = True

print("Custom Callback berhasil didefinisikan.")

Custom Callback berhasil didefinisikan.


In [7]:
X_dummy = np.random.rand(100, 10)
y_dummy = np.random.randint(0, 2, size=(100, 1))

print("Memulai simulasi training...")
model.fit(
    X_dummy,
    y_dummy,
    epochs=15,
    batch_size=8,
    callbacks=[RiskMonitoringCallback()],
    verbose=1
)

Memulai simulasi training...
Epoch 1/40
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.5500 - loss: 0.3997 - mae: 0.4881 
Epoch 2/40
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.5700 - loss: 0.3730 - mae: 0.4873 
Epoch 3/40
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.5400 - loss: 0.3412 - mae: 0.4632 
Epoch 4/40
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.5900 - loss: 0.3103 - mae: 0.4492 
Epoch 5/40
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6300 - loss: 0.2706 - mae: 0.4191 
Epoch 6/40
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6200 - loss: 0.2744 - mae: 0.4167 
Epoch 7/40
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6000 - loss: 0.2915 - mae: 0.4331 
Epoch 8/40
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6100 - loss: 0.2744 - mae: 0.4169 
Epoch 9/40
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6300 - loss: 0.2635 - mae: 0.4100 
Epoch 10/40
13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6700 - loss: 0.

In [8]:
save_path = 'models/tabular_model/'
if not os.path.exists(save_path):
    os.makedirs(save_path)

# Simpan model dalam format .keras
model_file = os.path.join(save_path, 'worksafe_model_v1.keras')
model.save(model_file)

print(f"SUKSES! Model disimpan di: {model_file}")

SUKSES! Model disimpan di: models/tabular_model/worksafe_model_v1.keras
